# Lesson 16b: RLHF Practical Implementation

**Implementations:**
- Reward model training from human preferences
- PPO-RLHF with KL penalty
- Direct Preference Optimization (DPO)
- Constitutional AI principles
- Complete RLHF pipeline

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from collections import deque
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Synthetic Environment for RLHF

We'll create a simple text generation task for demonstration.

In [ ]:
class SimpleTextEnv:
    """Simplified text generation environment.
    
    Task: Generate sequences that maximize "quality" (e.g., avoiding repetition,
    maintaining coherence). Human preferences will guide what constitutes quality.
    """
    
    def __init__(self, vocab_size=100, max_length=20):
        self.vocab_size = vocab_size
        self.max_length = max_length
        self.reset()
    
    def reset(self, prompt=None):
        """Reset with optional prompt."""
        if prompt is None:
            # Random prompt (first token)
            self.sequence = [np.random.randint(0, self.vocab_size)]
        else:
            self.sequence = list(prompt)
        return np.array(self.sequence)
    
    def step(self, action):
        """Add token to sequence."""
        self.sequence.append(action)
        done = len(self.sequence) >= self.max_length
        
        # Simple intrinsic reward (avoid repetition)
        reward = 0.0
        if action not in self.sequence[:-1]:  # New token
            reward = 0.1
        
        return np.array(self.sequence), reward, done
    
    def get_sequence(self):
        return self.sequence.copy()

print("Text environment created")

## 2. Language Model Policy

Simple autoregressive model for text generation.

In [ ]:
class LanguageModelPolicy(nn.Module):
    """Simple autoregressive language model."""
    
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        # Token embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # LSTM for sequence modeling
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        
        # Output head for next token prediction
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, sequence, hidden=None):
        """Forward pass.
        
        Args:
            sequence: [batch, seq_len] token IDs
            hidden: LSTM hidden state
        
        Returns:
            logits: [batch, seq_len, vocab_size]
            hidden: Updated hidden state
        """
        # Embed tokens
        embedded = self.embedding(sequence)  # [batch, seq_len, embed_dim]
        
        # LSTM forward
        lstm_out, hidden = self.lstm(embedded, hidden)  # [batch, seq_len, hidden_dim]
        
        # Predict next tokens
        logits = self.output(lstm_out)  # [batch, seq_len, vocab_size]
        
        return logits, hidden
    
    def get_log_probs(self, sequence, actions):
        """Get log probabilities for actions.
        
        Args:
            sequence: [batch, seq_len]
            actions: [batch, seq_len] next tokens
        
        Returns:
            log_probs: [batch, seq_len]
        """
        logits, _ = self.forward(sequence)
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Gather log probs for actual actions
        action_log_probs = log_probs.gather(2, actions.unsqueeze(-1)).squeeze(-1)
        return action_log_probs
    
    def sample(self, sequence, temperature=1.0):
        """Sample next token.
        
        Args:
            sequence: [batch, seq_len]
            temperature: Sampling temperature
        
        Returns:
            action: [batch] next tokens
            log_prob: [batch] log probabilities
        """
        logits, _ = self.forward(sequence)
        
        # Use last timestep logits
        logits = logits[:, -1, :] / temperature
        
        # Sample
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        
        return action, log_prob

print("Language model policy implemented")

## 3. Reward Model

Learns to predict human preferences via Bradley-Terry model.

In [ ]:
class RewardModel(nn.Module):
    """Reward model trained on preference comparisons."""
    
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        
        # Share architecture with LM (common in practice)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        
        # Scalar reward head
        self.reward_head = nn.Linear(hidden_dim, 1)
    
    def forward(self, sequence):
        """Compute scalar reward for sequence.
        
        Args:
            sequence: [batch, seq_len] token IDs
        
        Returns:
            reward: [batch] scalar rewards
        """
        # Embed and encode
        embedded = self.embedding(sequence)
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        
        # Use final hidden state for reward
        reward = self.reward_head(h_n[-1]).squeeze(-1)  # [batch]
        
        return reward

class PreferenceDataset:
    """Dataset of preference comparisons."""
    
    def __init__(self):
        self.comparisons = []  # List of (seq1, seq2, preference)
    
    def add_comparison(self, seq1, seq2, preference):
        """Add preference comparison.
        
        Args:
            seq1: First sequence
            seq2: Second sequence
            preference: 0 if seq1 preferred, 1 if seq2 preferred
        """
        self.comparisons.append((seq1, seq2, preference))
    
    def get_batch(self, batch_size):
        """Sample batch of comparisons."""
        indices = np.random.choice(len(self.comparisons), batch_size, replace=True)
        
        batch_seq1, batch_seq2, batch_prefs = [], [], []
        
        for idx in indices:
            seq1, seq2, pref = self.comparisons[idx]
            batch_seq1.append(seq1)
            batch_seq2.append(seq2)
            batch_prefs.append(pref)
        
        return batch_seq1, batch_seq2, batch_prefs
    
    def __len__(self):
        return len(self.comparisons)

def train_reward_model(reward_model, dataset, epochs=5, batch_size=32, lr=1e-4):
    """Train reward model on preference data.
    
    Uses Bradley-Terry model: P(seq1 > seq2) = sigmoid(r(seq1) - r(seq2))
    """
    optimizer = torch.optim.Adam(reward_model.parameters(), lr=lr)
    
    losses = []
    
    for epoch in range(epochs):
        epoch_losses = []
        
        # Multiple batches per epoch
        for _ in range(len(dataset) // batch_size):
            seq1_batch, seq2_batch, pref_batch = dataset.get_batch(batch_size)
            
            # Pad sequences to same length
            max_len = max(max(len(s) for s in seq1_batch), 
                         max(len(s) for s in seq2_batch))
            
            def pad_sequence(seq, max_len):
                return seq + [0] * (max_len - len(seq))
            
            seq1_padded = [pad_sequence(s, max_len) for s in seq1_batch]
            seq2_padded = [pad_sequence(s, max_len) for s in seq2_batch]
            
            seq1_tensor = torch.LongTensor(seq1_padded).to(device)
            seq2_tensor = torch.LongTensor(seq2_padded).to(device)
            pref_tensor = torch.FloatTensor(pref_batch).to(device)
            
            # Compute rewards
            r1 = reward_model(seq1_tensor)
            r2 = reward_model(seq2_tensor)
            
            # Bradley-Terry loss
            # P(seq2 > seq1) = sigmoid(r2 - r1)
            logits = r2 - r1
            loss = F.binary_cross_entropy_with_logits(logits, pref_tensor)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_losses.append(loss.item())
        
        avg_loss = np.mean(epoch_losses)
        losses.append(avg_loss)
        
        if (epoch + 1) % 1 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    return losses

print("Reward model implemented")

## 4. PPO-RLHF Implementation

PPO with learned reward model and KL penalty.

In [ ]:
class ValueNetwork(nn.Module):
    """Value function for PPO."""
    
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.value_head = nn.Linear(hidden_dim, 1)
    
    def forward(self, sequence):
        embedded = self.embedding(sequence)
        lstm_out, _ = self.lstm(embedded)
        values = self.value_head(lstm_out).squeeze(-1)  # [batch, seq_len]
        return values

class PPO_RLHF:
    """PPO with RLHF (reward model + KL penalty)."""
    
    def __init__(self, vocab_size, reward_model, kl_coef=0.05, 
                 gamma=0.99, eps_clip=0.2, lr=1e-5):
        # Policy (to be optimized)
        self.policy = LanguageModelPolicy(vocab_size).to(device)
        
        # Reference policy (frozen SFT)
        self.ref_policy = LanguageModelPolicy(vocab_size).to(device)
        self.ref_policy.load_state_dict(self.policy.state_dict())
        self.ref_policy.eval()
        for param in self.ref_policy.parameters():
            param.requires_grad = False
        
        # Value function
        self.value_net = ValueNetwork(vocab_size).to(device)
        
        # Reward model (frozen)
        self.reward_model = reward_model
        self.reward_model.eval()
        for param in self.reward_model.parameters():
            param.requires_grad = False
        
        # Hyperparameters
        self.kl_coef = kl_coef
        self.gamma = gamma
        self.eps_clip = eps_clip
        
        # Optimizers
        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr)
        self.value_optimizer = torch.optim.Adam(self.value_net.parameters(), lr=lr)
    
    def compute_rewards(self, sequences):
        """Compute RLHF rewards: r_theta(x,y) - beta * KL."""
        with torch.no_grad():
            # Reward model score
            rm_rewards = self.reward_model(sequences).cpu().numpy()
            
            # KL penalty (per token)
            # We'll compute KL divergence between policy and ref_policy
            policy_logits, _ = self.policy(sequences)
            ref_logits, _ = self.ref_policy(sequences)
            
            policy_probs = F.softmax(policy_logits, dim=-1)
            ref_probs = F.softmax(ref_logits, dim=-1)
            
            # KL(policy || ref) per token
            kl_div = (policy_probs * (policy_probs.log() - ref_probs.log())).sum(dim=-1)
            kl_penalty = kl_div.mean(dim=-1).cpu().numpy()  # [batch]
            
            # Total reward
            total_rewards = rm_rewards - self.kl_coef * kl_penalty
        
        return total_rewards, rm_rewards, kl_penalty
    
    def update(self, sequences, actions, old_log_probs, rewards, advantages):
        """PPO update step."""
        # Convert to tensors
        sequences = torch.LongTensor(sequences).to(device)
        actions = torch.LongTensor(actions).to(device)
        old_log_probs = torch.FloatTensor(old_log_probs).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        advantages = torch.FloatTensor(advantages).to(device)
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Policy loss (PPO clipped objective)
        new_log_probs = self.policy.get_log_probs(sequences, actions)
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        surr1 = ratio * advantages.unsqueeze(1)
        surr2 = torch.clamp(ratio, 1 - self.eps_clip, 1 + self.eps_clip) * advantages.unsqueeze(1)
        policy_loss = -torch.min(surr1, surr2).mean()
        
        # Value loss
        values = self.value_net(sequences)
        # Compute returns (simplified: just final reward)
        returns = rewards.unsqueeze(1).expand_as(values)
        value_loss = F.mse_loss(values, returns)
        
        # Update policy
        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.policy_optimizer.step()
        
        # Update value function
        self.value_optimizer.zero_grad()
        value_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.value_net.parameters(), 1.0)
        self.value_optimizer.step()
        
        return policy_loss.item(), value_loss.item()

print("PPO-RLHF implemented")

## 5. Direct Preference Optimization (DPO)

Train policy directly from preferences without explicit reward model.

In [ ]:
class DPO:
    """Direct Preference Optimization.
    
    Optimizes policy directly from preference data:
    L = -E[log sigmoid(beta * log(pi(y_w)/pi_ref(y_w)) - beta * log(pi(y_l)/pi_ref(y_l)))]
    """
    
    def __init__(self, vocab_size, beta=0.1, lr=1e-5):
        # Policy to optimize
        self.policy = LanguageModelPolicy(vocab_size).to(device)
        
        # Reference policy (frozen)
        self.ref_policy = LanguageModelPolicy(vocab_size).to(device)
        self.ref_policy.load_state_dict(self.policy.state_dict())
        self.ref_policy.eval()
        for param in self.ref_policy.parameters():
            param.requires_grad = False
        
        self.beta = beta
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr)
    
    def compute_sequence_logprob(self, policy, sequence):
        """Compute log probability of sequence under policy.
        
        Args:
            policy: Language model
            sequence: [batch, seq_len] token IDs
        
        Returns:
            log_prob: [batch] log probability of full sequence
        """
        # Get logits for all tokens
        logits, _ = policy(sequence[:, :-1])  # Predict next tokens
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Gather log probs for actual next tokens
        next_tokens = sequence[:, 1:]  # Shift by 1
        token_log_probs = log_probs.gather(2, next_tokens.unsqueeze(-1)).squeeze(-1)
        
        # Sum over sequence
        seq_log_prob = token_log_probs.sum(dim=-1)
        return seq_log_prob
    
    def update(self, seq_w, seq_l):
        """Update on preference comparison.
        
        Args:
            seq_w: Preferred (winning) sequences [batch, seq_len]
            seq_l: Dis-preferred (losing) sequences [batch, seq_len]
        """
        seq_w = torch.LongTensor(seq_w).to(device)
        seq_l = torch.LongTensor(seq_l).to(device)
        
        # Compute log probs under current policy
        log_pi_w = self.compute_sequence_logprob(self.policy, seq_w)
        log_pi_l = self.compute_sequence_logprob(self.policy, seq_l)
        
        # Compute log probs under reference policy
        with torch.no_grad():
            log_ref_w = self.compute_sequence_logprob(self.ref_policy, seq_w)
            log_ref_l = self.compute_sequence_logprob(self.ref_policy, seq_l)
        
        # DPO loss
        logits = self.beta * (log_pi_w - log_ref_w) - self.beta * (log_pi_l - log_ref_l)
        loss = -F.logsigmoid(logits).mean()
        
        # Update
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.optimizer.step()
        
        # Compute accuracy
        with torch.no_grad():
            accuracy = (logits > 0).float().mean().item()
        
        return loss.item(), accuracy

def train_dpo(dpo_agent, dataset, epochs=5, batch_size=16):
    """Train DPO on preference dataset."""
    losses = []
    accuracies = []
    
    for epoch in range(epochs):
        epoch_losses = []
        epoch_accs = []
        
        for _ in range(len(dataset) // batch_size):
            seq1_batch, seq2_batch, pref_batch = dataset.get_batch(batch_size)
            
            # Separate into preferred/dis-preferred
            seq_w, seq_l = [], []
            for s1, s2, pref in zip(seq1_batch, seq2_batch, pref_batch):
                if pref == 0:  # s1 preferred
                    seq_w.append(s1)
                    seq_l.append(s2)
                else:  # s2 preferred
                    seq_w.append(s2)
                    seq_l.append(s1)
            
            # Pad sequences
            max_len = max(max(len(s) for s in seq_w), max(len(s) for s in seq_l))
            
            def pad(seq, max_len):
                return seq + [0] * (max_len - len(seq))
            
            seq_w = [pad(s, max_len) for s in seq_w]
            seq_l = [pad(s, max_len) for s in seq_l]
            
            # Update
            loss, acc = dpo_agent.update(seq_w, seq_l)
            epoch_losses.append(loss)
            epoch_accs.append(acc)
        
        avg_loss = np.mean(epoch_losses)
        avg_acc = np.mean(epoch_accs)
        losses.append(avg_loss)
        accuracies.append(avg_acc)
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}")
    
    return losses, accuracies

print("DPO implemented")

## 6. Synthetic Preference Data Generation

Create preference data based on quality heuristics.

In [ ]:
def synthetic_preference_function(seq1, seq2):
    """Synthetic preference based on quality heuristics.
    
    Preferences:
    - Prefer diverse sequences (less repetition)
    - Prefer longer sequences (up to a point)
    - Prefer sequences with variety
    
    Returns:
        0 if seq1 preferred, 1 if seq2 preferred
    """
    def score_sequence(seq):
        # Diversity score (unique tokens / total tokens)
        diversity = len(set(seq)) / len(seq)
        
        # Length score (prefer 10-15 tokens)
        target_len = 12
        length_score = 1.0 - abs(len(seq) - target_len) / target_len
        length_score = max(0, length_score)
        
        # Repetition penalty
        max_repeat = 0
        for i in range(len(seq) - 1):
            if seq[i] == seq[i+1]:
                max_repeat += 1
        repetition_penalty = max_repeat / len(seq)
        
        # Total score
        score = diversity * 0.5 + length_score * 0.3 - repetition_penalty * 0.2
        return score
    
    score1 = score_sequence(seq1)
    score2 = score_sequence(seq2)
    
    # Add noise
    score1 += np.random.normal(0, 0.1)
    score2 += np.random.normal(0, 0.1)
    
    return 0 if score1 > score2 else 1

def generate_preference_dataset(env, policy, num_comparisons=1000):
    """Generate synthetic preference dataset."""
    dataset = PreferenceDataset()
    
    policy.eval()
    
    for _ in range(num_comparisons):
        # Generate two sequences from policy
        sequences = []
        for _ in range(2):
            state = env.reset()
            seq = [state[0]]
            done = False
            
            while not done and len(seq) < env.max_length:
                state_tensor = torch.LongTensor([seq]).to(device)
                with torch.no_grad():
                    action, _ = policy.sample(state_tensor, temperature=1.0)
                action = action.item()
                
                state, _, done = env.step(action)
                seq.append(action)
            
            sequences.append(seq)
        
        # Get preference
        pref = synthetic_preference_function(sequences[0], sequences[1])
        dataset.add_comparison(sequences[0], sequences[1], pref)
    
    return dataset

print("Preference data generation implemented")

## 7. Full RLHF Pipeline Example

In [ ]:
# Environment setup
env = SimpleTextEnv(vocab_size=50, max_length=15)

print("="*60)
print("RLHF Pipeline Demonstration")
print("="*60)

# Stage 1: Supervised Fine-Tuning (SFT)
print("\n[Stage 1] Supervised Fine-Tuning (simulated)")
sft_policy = LanguageModelPolicy(env.vocab_size).to(device)
print("✓ SFT policy initialized")

# Stage 2: Collect preferences and train reward model
print("\n[Stage 2] Reward Model Training")
print("Generating preference comparisons...")
pref_dataset = generate_preference_dataset(env, sft_policy, num_comparisons=500)
print(f"✓ Generated {len(pref_dataset)} preference comparisons")

print("\nTraining reward model...")
reward_model = RewardModel(env.vocab_size).to(device)
rm_losses = train_reward_model(reward_model, pref_dataset, epochs=3, batch_size=32)
print("✓ Reward model trained")

# Visualize reward model training
plt.figure(figsize=(10, 4))
plt.plot(rm_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Reward Model Training')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Reward Model Training Complete")
print("="*60)

## 8. PPO-RLHF Training Example

In [ ]:
# Stage 3: RL Fine-tuning with PPO
print("\n[Stage 3] PPO-RLHF Training")

ppo_agent = PPO_RLHF(env.vocab_size, reward_model, kl_coef=0.05)

# Training loop
num_iterations = 20
batch_size = 16

rm_rewards_history = []
kl_penalties_history = []
total_rewards_history = []

print("Training PPO-RLHF...")

for iteration in range(num_iterations):
    # Collect batch of sequences
    sequences = []
    actions_list = []
    log_probs_list = []
    
    for _ in range(batch_size):
        state = env.reset()
        seq = [state[0]]
        actions = []
        log_probs = []
        done = False
        
        while not done and len(seq) < env.max_length:
            state_tensor = torch.LongTensor([seq]).to(device)
            with torch.no_grad():
                action, log_prob = ppo_agent.policy.sample(state_tensor)
            action = action.item()
            
            state, _, done = env.step(action)
            seq.append(action)
            actions.append(action)
            log_probs.append(log_prob.item())
        
        sequences.append(seq)
        actions_list.append(actions)
        log_probs_list.append(log_probs)
    
    # Pad sequences
    max_len = max(len(s) for s in sequences)
    sequences_padded = [s + [0]*(max_len - len(s)) for s in sequences]
    actions_padded = [a + [0]*(max_len - 1 - len(a)) for a in actions_list]
    log_probs_padded = [lp + [0]*(max_len - 1 - len(lp)) for lp in log_probs_list]
    
    # Compute rewards
    seq_tensor = torch.LongTensor(sequences_padded).to(device)
    total_rewards, rm_rewards, kl_penalties = ppo_agent.compute_rewards(seq_tensor)
    
    # Advantages (simplified: just use rewards)
    advantages = total_rewards
    
    # Update
    policy_loss, value_loss = ppo_agent.update(
        sequences_padded, actions_padded, log_probs_padded,
        total_rewards, advantages
    )
    
    # Track metrics
    rm_rewards_history.append(rm_rewards.mean())
    kl_penalties_history.append(kl_penalties.mean())
    total_rewards_history.append(total_rewards.mean())
    
    if (iteration + 1) % 5 == 0:
        print(f"Iter {iteration+1}/{num_iterations} | "
              f"RM Reward: {rm_rewards.mean():.3f} | "
              f"KL: {kl_penalties.mean():.3f} | "
              f"Total: {total_rewards.mean():.3f}")

print("✓ PPO-RLHF training complete")

# Visualize training
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(rm_rewards_history)
axes[0].set_title('Reward Model Score')
axes[0].set_xlabel('Iteration')
axes[0].grid(True)

axes[1].plot(kl_penalties_history)
axes[1].set_title('KL Penalty')
axes[1].set_xlabel('Iteration')
axes[1].grid(True)

axes[2].plot(total_rewards_history)
axes[2].set_title('Total Reward (RM - KL)')
axes[2].set_xlabel('Iteration')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 9. DPO Training Example

In [ ]:
print("\n" + "="*60)
print("Direct Preference Optimization (DPO) Training")
print("="*60)

# Initialize DPO agent
dpo_agent = DPO(env.vocab_size, beta=0.1)

# Train on same preference dataset
print("\nTraining DPO...")
dpo_losses, dpo_accs = train_dpo(dpo_agent, pref_dataset, epochs=5, batch_size=16)

print("✓ DPO training complete")

# Visualize DPO training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(dpo_losses)
axes[0].set_title('DPO Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

axes[1].plot(dpo_accs)
axes[1].set_title('DPO Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 10. Comparison: RLHF vs DPO

In [ ]:
def evaluate_policy(policy, env, reward_model, num_episodes=50):
    """Evaluate policy by generating sequences and computing reward model scores."""
    policy.eval()
    rewards = []
    diversities = []
    lengths = []
    
    for _ in range(num_episodes):
        state = env.reset()
        seq = [state[0]]
        done = False
        
        while not done and len(seq) < env.max_length:
            state_tensor = torch.LongTensor([seq]).to(device)
            with torch.no_grad():
                action, _ = policy.sample(state_tensor)
            action = action.item()
            
            state, _, done = env.step(action)
            seq.append(action)
        
        # Compute reward
        seq_tensor = torch.LongTensor([seq]).to(device)
        with torch.no_grad():
            reward = reward_model(seq_tensor).item()
        
        rewards.append(reward)
        diversities.append(len(set(seq)) / len(seq))
        lengths.append(len(seq))
    
    return {
        'reward_mean': np.mean(rewards),
        'reward_std': np.std(rewards),
        'diversity': np.mean(diversities),
        'length': np.mean(lengths)
    }

print("\n" + "="*60)
print("Evaluation Comparison")
print("="*60)

# Evaluate all policies
print("\nEvaluating SFT baseline...")
sft_results = evaluate_policy(sft_policy, env, reward_model)

print("Evaluating PPO-RLHF policy...")
ppo_results = evaluate_policy(ppo_agent.policy, env, reward_model)

print("Evaluating DPO policy...")
dpo_results = evaluate_policy(dpo_agent.policy, env, reward_model)

# Display results
print("\n" + "="*60)
print("Results Summary")
print("="*60)

print(f"\nSFT Baseline:")
print(f"  Reward: {sft_results['reward_mean']:.3f} ± {sft_results['reward_std']:.3f}")
print(f"  Diversity: {sft_results['diversity']:.3f}")
print(f"  Avg Length: {sft_results['length']:.1f}")

print(f"\nPPO-RLHF:")
print(f"  Reward: {ppo_results['reward_mean']:.3f} ± {ppo_results['reward_std']:.3f}")
print(f"  Diversity: {ppo_results['diversity']:.3f}")
print(f"  Avg Length: {ppo_results['length']:.1f}")
print(f"  Improvement: {((ppo_results['reward_mean'] - sft_results['reward_mean']) / abs(sft_results['reward_mean']) * 100):.1f}%")

print(f"\nDPO:")
print(f"  Reward: {dpo_results['reward_mean']:.3f} ± {dpo_results['reward_std']:.3f}")
print(f"  Diversity: {dpo_results['diversity']:.3f}")
print(f"  Avg Length: {dpo_results['length']:.1f}")
print(f"  Improvement: {((dpo_results['reward_mean'] - sft_results['reward_mean']) / abs(sft_results['reward_mean']) * 100):.1f}%")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

methods = ['SFT', 'PPO-RLHF', 'DPO']
rewards = [sft_results['reward_mean'], ppo_results['reward_mean'], dpo_results['reward_mean']]
diversities = [sft_results['diversity'], ppo_results['diversity'], dpo_results['diversity']]
lengths = [sft_results['length'], ppo_results['length'], dpo_results['length']]

axes[0].bar(methods, rewards)
axes[0].set_title('Reward Model Score')
axes[0].set_ylabel('Score')
axes[0].grid(True, alpha=0.3)

axes[1].bar(methods, diversities)
axes[1].set_title('Sequence Diversity')
axes[1].set_ylabel('Diversity')
axes[1].grid(True, alpha=0.3)

axes[2].bar(methods, lengths)
axes[2].set_title('Average Sequence Length')
axes[2].set_ylabel('Length')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Constitutional AI Example

In [ ]:
class ConstitutionalAI:
    """Constitutional AI with principle-based preference generation."""
    
    def __init__(self, principles):
        """
        Args:
            principles: List of scoring functions that evaluate sequences
        """
        self.principles = principles
    
    def evaluate_sequence(self, sequence):
        """Evaluate sequence against all principles."""
        scores = [principle(sequence) for principle in self.principles]
        return np.mean(scores)
    
    def generate_preference(self, seq1, seq2):
        """Generate preference based on principles.
        
        Returns:
            0 if seq1 preferred, 1 if seq2 preferred
        """
        score1 = self.evaluate_sequence(seq1)
        score2 = self.evaluate_sequence(seq2)
        return 0 if score1 > score2 else 1

# Define principles
def principle_diversity(seq):
    """Principle: Prefer diverse sequences."""
    return len(set(seq)) / len(seq)

def principle_no_repetition(seq):
    """Principle: Avoid immediate repetition."""
    repeats = sum(1 for i in range(len(seq)-1) if seq[i] == seq[i+1])
    return 1.0 - (repeats / len(seq))

def principle_reasonable_length(seq):
    """Principle: Prefer reasonable length (10-15 tokens)."""
    target = 12
    return 1.0 - min(abs(len(seq) - target) / target, 1.0)

# Constitutional AI with principles
print("\n" + "="*60)
print("Constitutional AI Example")
print("="*60)

principles = [
    principle_diversity,
    principle_no_repetition,
    principle_reasonable_length
]

constitutional_ai = ConstitutionalAI(principles)

# Generate preference dataset using constitutional principles
print("\nGenerating preferences from constitutional principles...")
constitutional_dataset = PreferenceDataset()

for _ in range(500):
    # Generate two random sequences
    sequences = []
    for _ in range(2):
        state = env.reset()
        seq = [state[0]]
        for _ in range(np.random.randint(5, 15)):
            action = np.random.randint(0, env.vocab_size)
            seq.append(action)
        sequences.append(seq)
    
    # Get preference from constitutional AI
    pref = constitutional_ai.generate_preference(sequences[0], sequences[1])
    constitutional_dataset.add_comparison(sequences[0], sequences[1], pref)

print(f"✓ Generated {len(constitutional_dataset)} constitutional preferences")

# Train policy on constitutional preferences
print("\nTraining policy on constitutional preferences (using DPO)...")
constitutional_dpo = DPO(env.vocab_size, beta=0.1)
const_losses, const_accs = train_dpo(constitutional_dpo, constitutional_dataset, 
                                     epochs=5, batch_size=16)

print("✓ Constitutional AI policy trained")

# Evaluate
print("\nEvaluating constitutional policy...")
const_results = evaluate_policy(constitutional_dpo.policy, env, reward_model)

print(f"\nConstitutional AI Policy:")
print(f"  Reward: {const_results['reward_mean']:.3f} ± {const_results['reward_std']:.3f}")
print(f"  Diversity: {const_results['diversity']:.3f}")
print(f"  Avg Length: {const_results['length']:.1f}")

## Summary

### Implemented:

1. **Reward Model Training**
   - Bradley-Terry preference learning
   - Cross-entropy loss on comparisons
   - Evaluation on held-out preferences

2. **PPO-RLHF**
   - Policy optimization with learned rewards
   - KL penalty to reference policy
   - Value function for advantage estimation
   - Clipped PPO objective

3. **Direct Preference Optimization (DPO)**
   - Direct policy learning from preferences
   - No explicit reward model
   - More stable than PPO-RLHF
   - Simpler implementation

4. **Constitutional AI**
   - Principle-based preference generation
   - AI feedback instead of human
   - More scalable and consistent

### Key Insights:

1. **RLHF Pipeline**: Three stages (SFT → RM → RL) all critical
2. **KL Penalty**: Essential for preventing reward hacking
3. **DPO**: Simpler alternative to PPO-RLHF with similar performance
4. **Data Quality**: Preference quality determines final policy quality
5. **Principles**: Explicit principles (Constitutional AI) improve transparency

### Real-World Applications:

- **ChatGPT**: Trained with RLHF for helpfulness and safety
- **Claude**: Uses Constitutional AI principles
- **GitHub Copilot**: RLHF for code quality
- **Robotics**: Learning from human demonstrations

### Next Steps:

**Professional Practice Series (X1-X4)**:
- Deployment and production systems
- Hyperparameter tuning at scale
- Debugging and monitoring
- Real-world applications and case studies